In [ ]:
# https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma4_(E2B)_Reinforcement_Learning_Sudoku_Game.ipynb#scrollTo=hzPgFeIkZn9q
# https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide

In [ ]:
#%%capture
#import os, importlib.util
#!pip install --upgrade -qqq uv
#if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
#    try: import numpy, PIL; _numpy = fs"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
#    except: _numpy = "numpy"; _pil = "pillow"
#    # Gemma 4 requires transformers >= 5.5.0 — do NOT pin to 4.x here
#    !uv pip install -qqq \
#        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
#        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
#        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
#        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
#elif importlib.util.find_spec("unsloth") is None:
#    !uv pip install -qqq unsloth
## Gemma 4 requires transformers >= 5.5.0
#!uv pip install --upgrade --no-deps "transformers>=5.5.0" tokenizers "trl>=0.28.0" unsloth unsloth_zoo

In [ ]:
#!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

In [ ]:
from unsloth import FastVisionModel
import torch
max_seq_length = 4096 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = False, # Enable vLLM fast inference
)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)